# CareCompass — Healthcare Symptom Triage & Appointment Assistant

**Capstone notebook | LangGraph + RAG + OpenAI**

This notebook demonstrates the CareCompass agent pipeline using the project's existing `agent.py`, `rag_engine.py`, `guidelines.json`, and `slots.json` files from GitHub.

> **Safety notice:** CareCompass is an educational, non-diagnostic demo. It is not a medical device and must not be used to make medical decisions. Emergency phrase detection is a simple rule-based safeguard, not a guarantee that emergencies will be detected. If someone may be in immediate danger, contact local emergency services.

## What this notebook demonstrates
1. Set up the project in Google Colab.
2. Install the project dependencies.
3. Configure an OpenAI API key without hard-coding it.
4. Inspect the LangGraph workflow and test the deterministic emergency check.
5. Run sample requests through the complete agent pipeline.
6. Review RAG sources and simulated appointment results.

**Costs:** Running the LLM and embedding cells uses your OpenAI API account and may incur charges. The deterministic emergency-check test does not require an API call.

## 1. Clone the GitHub repository

The repository is public, so no GitHub token is needed. If your repository URL changes, update `REPO_URL` below.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/ashkaa-001/CareCompass.git"
PROJECT_DIR = Path("/content/CareCompass")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print("Repository folder already exists:", PROJECT_DIR)

os.chdir(PROJECT_DIR)
print("Project directory:", Path.cwd())
print("Files:", sorted(p.name for p in Path.cwd().iterdir() if p.is_file()))

## 2. Install dependencies

This uses the repository's `requirements.txt` so the notebook stays aligned with the deployed project.

In [ ]:
%pip install -r requirements.txt

## 3. Configure the OpenAI API key

Enter your own API key when prompted. `getpass` hides the key as you type; do not print it or save it in a notebook cell. Never commit API keys to GitHub.

In [ ]:
from getpass import getpass
import os

api_key = getpass("Enter your OpenAI API key (input hidden): ").strip()
if not api_key:
    raise ValueError("An OpenAI API key is required to run the LLM and RAG examples.")

os.environ["OPENAI_API_KEY"] = api_key
print("API key configured for this Colab session.")

## 4. Inspect the project files

The RAG component reads public-health guidance from `guidelines.json`; the scheduling component reads sample appointment slots from `slots.json`. The slots are simulated and do not create real bookings.

In [ ]:
import json
from pathlib import Path

required_files = ["agent.py", "rag_engine.py", "guidelines.json", "slots.json"]
missing = [name for name in required_files if not Path(name).exists()]
if missing:
    raise FileNotFoundError(f"Missing project files: {missing}")

with open("guidelines.json", "r", encoding="utf-8") as f:
    guidelines = json.load(f)
with open("slots.json", "r", encoding="utf-8") as f:
    slots = json.load(f)

print(f"Guideline records available: {len(guidelines)}")
print("Guideline fields:", list(guidelines[0].keys()) if guidelines else "No records")
print("\nSimulated appointment slot categories:", list(slots.keys()))

## 5. Load the LangGraph workflow

The graph checks for configured emergency phrases first. A detected positive emergency indicator routes directly to a fixed escalation response, bypassing the LLM for that path. Non-emergency requests proceed to the router and relevant downstream agent.

In [ ]:
from agent import app as carecompass_graph
from agent import check_emergency

print("LangGraph workflow loaded.")
print("Workflow nodes:")
for node_name in carecompass_graph.get_graph().nodes:
    print("-", node_name)

## 6. Test deterministic emergency screening (no LLM call)

This cell tests the rule-based phrase check by itself. These are **fictional test examples**, not medical advice. A keyword check can miss emergencies or flag a phrase out of context, so it must never be treated as a complete clinical safety system.

In [ ]:
emergency_test_cases = [
    ("I have severe chest pain.", True),
    ("I cannot breathe.", True),
    ("I have a mild headache.", False),
]

for message, expected in emergency_test_cases:
    result = check_emergency({"query": message})
    actual = result["emergency_detected"]
    status = "PASS" if actual == expected else "CHECK"
    print(f"{status} | expected={expected} | detected={actual} | input={message!r}")
    print("  triggers:", result.get("emergency_triggers", []))

## 7. Run one request through the complete agent pipeline

Edit `query` to try a different **fictional, non-identifying** example. This call may use the OpenAI chat model and, for self-care responses, the embedding model. It may incur API charges.

If an emergency phrase is detected, the graph should return the fixed escalation response without needing an LLM call for that path.

In [ ]:
query = "I have had a mild headache since this morning. What should I consider doing?"

initial_state = {
    "query": query,
    "api_key": api_key,
    "intent": "",
    "symptoms": [],
    "duration": "not specified",
    "severity": "not specified",
    "urgency": "",
    "emergency_detected": False,
    "emergency_triggers": [],
    "emergency_negated": [],
    "recommended_action": "",
    "retrieved_context": [],
    "appointment_type": "",
    "appointment_slot": "",
    "response": "",
    "sources": [],
}

result = carecompass_graph.invoke(initial_state)

print("USER QUERY")
print(query)
print("\nINTENT:", result.get("intent", "not specified"))
print("SYMPTOMS:", result.get("symptoms", []))
print("DURATION:", result.get("duration", "not specified"))
print("SEVERITY:", result.get("severity", "not specified"))
print("URGENCY:", result.get("urgency", "not specified"))
print("\nCARECOMPASS RESPONSE")
print(result.get("response", "No response returned."))
print("\nSOURCES")
for source in result.get("sources", []):
    print("-", source)

## 8. Try additional sample requests

Run these one at a time. These examples are for software demonstration only; do not use real personal health information. Each full workflow run may incur API charges.

In [ ]:
def run_carecompass(query_text):
    state = {
        "query": query_text,
        "api_key": api_key,
        "intent": "",
        "symptoms": [],
        "duration": "not specified",
        "severity": "not specified",
        "urgency": "",
        "emergency_detected": False,
        "emergency_triggers": [],
        "emergency_negated": [],
        "recommended_action": "",
        "retrieved_context": [],
        "appointment_type": "",
        "appointment_slot": "",
        "response": "",
        "sources": [],
    }
    output = carecompass_graph.invoke(state)
    print("Query:", query_text)
    print("Intent:", output.get("intent", "not specified"))
    print("Urgency:", output.get("urgency", "not specified"))
    print("\nResponse:\n", output.get("response", "No response returned."))
    if output.get("sources"):
        print("\nSources:")
        for source in output["sources"]:
            print("-", source)
    if output.get("appointment_slot"):
        print("\nAppointment slot:", output["appointment_slot"])
        print("(Simulated only — no real booking was made.)")
    return output

# Example A: general self-care information
# run_carecompass("What are general ways to stay hydrated?")

# Example B: simulated appointment request
# run_carecompass("I would like to schedule a general doctor appointment.")

# Example C: out-of-scope request
# run_carecompass("Tell me exactly what disease I have and prescribe medicine.")

## 9. Understand the RAG pipeline

CareCompass uses a hybrid retrieval approach:

1. `guidelines.json` supplies the public-health guidance records.
2. BM25 ranks records using keyword overlap.
3. ChromaDB retrieves semantically similar records using OpenAI `text-embedding-3-small` embeddings.
4. Reciprocal-rank fusion combines the two ranked lists.
5. The Knowledge Agent uses retrieved context to prepare a general-information response and returns source names.

The vector index is generated locally in `chroma_db/` on first retrieval. In a fresh Colab runtime, the index may need to be built again, which uses the OpenAI embeddings API.

In [ ]:
from rag_engine import hybrid_search

# This direct retrieval test calls the embeddings API and may incur charges.
retrieved = hybrid_search(
    "general information about staying hydrated",
    top_k=3,
    api_key=api_key,
)

for i, doc in enumerate(retrieved, start=1):
    print(f"{i}. {doc.get('title', 'Untitled')}")
    print("   Topic:", doc.get("topic", ""))
    print("   Source:", doc.get("source", ""))
    print("   URL:", doc.get("source_url", ""))
    print()

## 10. Safety, scope, and limitations

- **Non-diagnostic:** The system is educational and does not diagnose conditions or prescribe medication.
- **Emergency handling:** A deterministic phrase check routes recognized positive emergency indicators to a fixed response. It can still miss emergencies, and it is not a substitute for emergency services.
- **Evidence-grounded guidance:** The RAG workflow retrieves records from the project's `guidelines.json`. Retrieval quality depends on the contents and relevance of those records.
- **Simulated scheduling:** Appointment details are read from `slots.json`; no appointment is actually booked and no clinic is contacted.
- **API key security:** The key is entered at runtime and should never be written into notebook source or committed to GitHub.
- **Privacy:** Use fictional examples in this public notebook. Do not enter names, contact details, or identifiable health information.
- **Runtime storage:** Colab files and the generated ChromaDB index may be lost when the runtime resets.

## 11. Suggested evaluation checklist

Record the actual result of each test before presenting it as passed.

| Test | Expected behavior |
|---|---|
| Recognized emergency phrase | Fixed escalation response; LLM bypassed on this path |
| Ordinary self-care question | Relevant retrieved guidance and source names, where available |
| Appointment request | Sample slot clearly identified as simulated |
| Diagnosis or prescription request | Scope-limitation response |
| Missing or invalid API key | Clear error or request to provide a valid key |

## Conclusion

This notebook demonstrates the same LangGraph workflow used by the CareCompass application, with hybrid RAG retrieval and simulated appointment scheduling. The Streamlit interface is deployed separately on Render.

**Project links**
- GitHub: https://github.com/ashkaa-001/CareCompass
- Live demo: https://carecompass-r3sx.onrender.com/

*CareCompass is a student capstone demonstration, not a medical service.*